In [2]:
# Cell 1: Install dependencies
!pip install sentence-transformers chromadb numpy

     |████████████████████████████████| 255 kB 24.9 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 97.4 MB/s eta 0:00:01
     |████████████████████████████████| 34.5 MB 111.9 MB/s eta 0:00:01
     |████████████████████████████████| 10.0 MB 85.4 MB/s eta 0:00:01
     |████████████████████████████████| 471 kB 121.7 MB/s eta 0:00:01
ERROR: transformers 4.46.3 has requirement tokenizers<0.21,>=0.20, but you'll have tokenizers 0.19.1 which is incompatible.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
pip install streamlit chromadb sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 116.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install --upgrade ipywidgets tqdm

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.0 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.66.4
    Uninstalling tqdm-4.66.4:
      Successfully uninstalled tqdm-4.66.4
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [7]:
!pip install sentence-transformers chromadb fastapi uvicorn nest_asyncio

In [1]:
import os

# 1. Disable Hugging Face interactive progress bar (avoids tqdm FloatProgress crash)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. Suppress TensorFlow GPU warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize ChromaDB in-memory client
client = chromadb.Client()

# Specify the full Hugging Face repository tag
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    normalize_embeddings=True
)

# Create collection with cosine distance metric
collection = client.create_collection(
    name="legal_cases_db",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"}
)

/home/labuser/.local/lib/python3.8/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:


import chromadb
from chromadb.utils import embedding_functions
import numpy as np

# Step 1 & 2: Define legal dataset chunks
legal_cases = [
    {
        "id": "case_001",
        "title": "TechCorp vs. DataFlow (2021)",
        "content": "Breach of Non-Disclosure Agreement (NDA). Defendant shared trade secrets regarding machine learning proprietary algorithms with a competitor. Court ruled in favor of Plaintiff awarding $2.5M in damages.",
        "outcome": "Plaintiff Won - Damages Awarded"
    },
    {
        "id": "case_002",
        "title": "Apex Logistics vs. TransGlobal (2019)",
        "content": "Breach of commercial supply contract due to delivery delays. Force majeure clause cited due to extreme weather conditions. The court held that weather did not fully justify total non-performance.",
        "outcome": "Partial Settlement - Defendant Liable"
    },
    {
        "id": "case_003",
        "title": "InnovateLabs vs. Former Employee (2022)",
        "content": "Enforceability of non-compete covenant and theft of confidential source code. Injunction granted prohibiting defendant from working for direct competitor for 12 months.",
        "outcome": "Injunction Granted"
    },
    {
        "id": "case_004",
        "title": "Urban Properties vs. Retail Chain (2020)",
        "content": "Commercial lease dispute over unpaid rent during pandemic shutdowns. Frustration of purpose doctrine rejected; tenant required to pay back rent under restructured timeline.",
        "outcome": "Defendant Liable for Back Rent"
    }
]

# Step 3: Initialize Vector DB with all-MiniLM-L6-v2
client = chromadb.Client()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2",
    normalize_embeddings=True  # Enables cosine similarity calculation directly
)

# ChromaDB collection configured for cosine distance
try:
    client.delete_collection("legal_cases_db")
except Exception:
    pass

collection = client.create_collection(
    name="legal_cases_db",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[item["id"] for item in legal_cases],
    documents=[item["content"] for item in legal_cases],
    metadatas=[{"title": item["title"], "outcome": item["outcome"]} for item in legal_cases]
)

# Step 4, 5 & 6: Query and return top-k matches with % similarity
def search_legal_precedents(query_text: str, top_k: int = 2):
    results = collection.query(
        query_texts=[query_text],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    output = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        # For normalized vectors in cosine space: distance = 1 - cosine_similarity
        similarity_score = max(0.0, 1.0 - dist)
        similarity_percentage = round(similarity_score * 100, 2)
        
        output.append({
            "case_title": meta["title"],
            "summary": doc,
            "predicted_outcome": meta["outcome"],
            "similarity": f"{similarity_percentage}%"
        })
    return output

# Run Query
test_query = "Employee leaked proprietary AI code to a competing software firm under an NDA."
results = search_legal_precedents(test_query, top_k=3)

for idx, res in enumerate(results, 1):
    print(f"Rank {idx}: {res['case_title']}")
    print(f"Similarity: {res['similarity']}")
    print(f"Precedent Outcome: {res['predicted_outcome']}")
    print(f"Summary: {res['summary']}\n" + "-"*50)

Rank 1: TechCorp vs. DataFlow (2021)
Similarity: 67.66%
Precedent Outcome: Plaintiff Won - Damages Awarded
Summary: Breach of Non-Disclosure Agreement (NDA). Defendant shared trade secrets regarding machine learning proprietary algorithms with a competitor. Court ruled in favor of Plaintiff awarding $2.5M in damages.
--------------------------------------------------
Rank 2: InnovateLabs vs. Former Employee (2022)
Similarity: 41.98%
Precedent Outcome: Injunction Granted
Summary: Enforceability of non-compete covenant and theft of confidential source code. Injunction granted prohibiting defendant from working for direct competitor for 12 months.
--------------------------------------------------
Rank 3: Apex Logistics vs. TransGlobal (2019)
Similarity: 14.64%
Precedent Outcome: Partial Settlement - Defendant Liable
Summary: Breach of commercial supply contract due to delivery delays. Force majeure clause cited due to extreme weather conditions. The court held that weather did not fully 

In [12]:
import threading
import asyncio
import uvicorn
import nest_asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import chromadb
from chromadb.utils import embedding_functions
from IPython.display import display, HTML

nest_asyncio.apply()

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

client = chromadb.Client()
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2",
    normalize_embeddings=True
)
collection = client.get_or_create_collection(
    name="legal_api_db",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"}
)

if collection.count() == 0:
    collection.add(
        ids=["1", "2"],
        documents=[
            "Ex-employee violated NDA by exporting proprietary model weights to competitor.",
            "Breach of supply agreement due to logistics delay; force majeure rejected."
        ],
        metadatas=[
            {"title": "TechCorp v. DataFlow", "outcome": "Plaintiff Won"},
            {"title": "Apex v. TransGlobal", "outcome": "Defendant Liable"}
        ]
    )

class QueryRequest(BaseModel):
    query: str
    top_k: int = 2

@app.post("/search")
def search(req: QueryRequest):
    res = collection.query(query_texts=[req.query], n_results=req.top_k)
    output = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        output.append({
            "title": meta["title"],
            "outcome": meta["outcome"],
            "summary": doc,
            "similarity": f"{round(max(0.0, 1.0 - dist) * 100, 2)}%"
        })
    return {"results": output}

def start_server():
    # Create and set a new event loop for this background thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    
    config = uvicorn.Config(app=app, host="127.0.0.1", port=8000, log_level="error")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

# Start the server daemon thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Render HTML + JavaScript interface directly in the cell output
display(HTML("""
<div style="font-family: sans-serif; max-width: 600px; padding: 10px;">
    <h3>Legal Outcome Search (HTML + JS calling FastAPI)</h3>
    <textarea id="queryBox" style="width: 100%; height: 60px;">Employee leaked model code under an NDA</textarea><br><br>
    <button onclick="fetchResults()" style="padding: 8px 16px; background: #007bff; color: white; border: none; border-radius: 4px; cursor: pointer;">Search API</button>
    <div id="results" style="margin-top: 15px;"></div>
</div>

<script>
async function fetchResults() {
    const query = document.getElementById('queryBox').value;
    const resDiv = document.getElementById('results');
    resDiv.innerHTML = "Searching...";
    
    try {
        const response = await fetch('http://127.0.0.1:8000/search', {
            method: 'POST',
            headers: {'Content-Type': 'application/json'},
            body: JSON.stringify({query: query, top_k: 2})
        });
        const data = await response.json();
        resDiv.innerHTML = data.results.map(r => `
            <div style="border: 1px solid #ccc; padding: 10px; margin-bottom: 8px; border-radius: 4px;">
                <b>${r.title}</b> (Similarity: ${r.similarity})<br>
                <span>Outcome: <i>${r.outcome}</i></span><br>
                <small>${r.summary}</small>
            </div>
        `).join('');
    } catch (err) {
        resDiv.innerHTML = "Error connecting to FastAPI backend: " + err;
    }
}
</script>
"""))